# Gold Mart Creation

This notebook creates the approved Gold star-schema tables from the trusted Silver layer.

Gold creation is separated from technical verification, which is maintained in `04_gold_mart_validation.ipynb`.

## Purpose and Grain

Gold contains four reusable dimensions and one fact table:

- `dim_date`
- `dim_time`
- `dim_taxi_zone`
- `dim_weather_hour`
- `fact_green_taxi_trip`

The fact grain is one accepted Silver Green Taxi source record. No Gold-side deduplication is applied without an approved upstream identity rule.

## Silver Source Inspection

Gold uses the following trusted Silver source tables:

- `green_taxi`
- `taxi_zones`
- `weather`

The source inspection cells document the available columns, data types, and representative records used by the Gold model.

In [0]:
SHOW TABLES IN `ftw-week-08`.`02_silver`;

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.green_taxi;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.green_taxi
LIMIT 10;

## Green Taxi Table Structure

The `green_taxi` Silver table contains the accepted trip-level source records, including timestamps, location identifiers, distance, duration, passenger count, financial measures, DQ flags, and lineage.

It is the source of `fact_green_taxi_trip`.

## Green Taxi Column Inventory

Gold uses the timestamps, LocationIDs, trip measures, financial measures, DQ flags, and source identifiers retained in Silver.

Because the source has no guaranteed business trip identifier, Gold uses a deterministic technical fact-row key based on stable source attributes. It is not treated as a real-world trip business key.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.taxi_zones;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.taxi_zones
LIMIT 10;

## Taxi Zone Table Structure

The `taxi_zones` Silver table provides one trusted record per `LocationID`. Gold uses it to build the reusable pickup/drop-off `dim_taxi_zone` role-playing dimension.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.weather;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.weather
ORDER BY weather_datetime
LIMIT 10;

## Weather Column Inventory and Preview

The `weather` Silver table provides trusted NYC local-hour observations identified by `weather_datetime` (`TIMESTAMP_NTZ`). Gold derives `weather_hour_key` from that local hour and retains the approved measurements and source context.

## Final Gold Dimensional Model

The approved star schema contains four dimensions and one fact table.

The fact table stores one accepted Silver Green Taxi source record per row. Date, time, and taxi-zone dimensions play separate pickup and drop-off roles, while Weather is associated with the pickup local hour.

## Gold Model Conventions

- `dim_time` has 25 members: key `0` for Unknown and keys `1–24` for hours `00:00–23:00`.
- `dim_weather_hour` has one Unknown member plus one member for every accepted Silver weather hour.
- Taxi Zone `LocationID` is reused as the dimension key for this assignment.
- The fact retains all accepted Silver rows and their lineage/DQ context.
- March–May business analysis filters `dq_out_of_range_datetime = FALSE`; the underlying Gold fact remains complete.

## Build Order

Create the dimensions first, then create the fact table so all foreign-key lookups are available:

1. `dim_date`
2. `dim_time`
3. `dim_taxi_zone`
4. `dim_weather_hour`
5. `fact_green_taxi_trip`

In [0]:
CREATE SCHEMA IF NOT EXISTS `ftw-week-08`.`03_gold`;

## Create DIM_DATE

`dim_date` is generated from the minimum and maximum non-null dates found across both accepted pickup and drop-off timestamps.

A continuous calendar row is created for every date in that range. `is_holiday` remains `NULL` until a verified holiday reference is approved.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_date
USING DELTA
AS
WITH required_dates AS (

    SELECT TO_DATE(lpep_pickup_datetime) AS full_date
    FROM `ftw-week-08`.`02_silver`.green_taxi

    UNION

    SELECT TO_DATE(lpep_dropoff_datetime) AS full_date
    FROM `ftw-week-08`.`02_silver`.green_taxi

),

date_range AS (
    SELECT
        MIN(full_date) AS min_date,
        MAX(full_date) AS max_date
    FROM required_dates
    WHERE full_date IS NOT NULL
),

dates AS (
    SELECT EXPLODE(
        SEQUENCE(min_date, max_date, INTERVAL 1 DAY)
    ) AS full_date
    FROM date_range
)

SELECT
    CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT) AS date_key,
    full_date,
    YEAR(full_date) AS year,
    QUARTER(full_date) AS quarter,
    MONTH(full_date) AS month_number,
    DATE_FORMAT(full_date, 'MMMM') AS month_name,
    DAYOFMONTH(full_date) AS day_of_month,
    DATE_FORMAT(full_date, 'EEEE') AS day_name,
    DAYOFWEEK(full_date) AS day_of_week,
    DAYOFWEEK(full_date) IN (1, 7) AS is_weekend,
    CAST(NULL AS BOOLEAN) AS is_holiday
FROM dates;

## Create DIM_TIME

`dim_time` contains one Unknown member plus one member for each hour `00:00–23:00`. It supports the pickup-time and drop-off-time roles in the fact table.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_time
USING DELTA
AS
SELECT
    0 AS time_key,
    CAST(NULL AS INT) AS hour_24,
    'Unknown' AS hour_label,
    'Unknown' AS day_period

UNION ALL

SELECT
    hour_24 + 1 AS time_key,
    hour_24,
    CONCAT(
        LPAD(CAST(hour_24 AS STRING), 2, '0'),
        ':00'
    ) AS hour_label,
    CASE
        WHEN hour_24 < 12 THEN 'AM'
        ELSE 'PM'
    END AS day_period
FROM (
    SELECT EXPLODE(SEQUENCE(0, 23)) AS hour_24
);

## Create DIM_TAXI_ZONE

`dim_taxi_zone` contains one trusted Silver Taxi Zone `LocationID` per row and supports both pickup and drop-off location roles.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_taxi_zone
USING DELTA
AS
SELECT
    CAST(LocationID AS INT) AS taxi_zone_key,
    CAST(LocationID AS INT) AS location_id,
    Borough AS borough,
    Zone AS zone_name,
    service_zone
FROM `ftw-week-08`.`02_silver`.taxi_zones;

## Create DIM_WEATHER_HOUR

The dimension grain is one accepted Silver local-hour observation, plus one documented Unknown member. `weather_hour_key` is the primary key; `weather_timestamp_local` retains the `TIMESTAMP_NTZ` business timestamp.

## DIM_WEATHER_HOUR Build and Unknown Weather Handling

The dimension retains every accepted Silver Weather hour and adds one Unknown member at `weather_hour_key = 0`. The Unknown member represents a pickup hour without an available observation and contains no assumed Weather measurements.

The source-to-Gold validation reports the retained hourly count and the separate Unknown member.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_weather_hour
USING DELTA
AS
SELECT
    CAST(0 AS BIGINT) AS weather_hour_key,
    CAST(NULL AS TIMESTAMP_NTZ) AS weather_timestamp_local,
    CAST(NULL AS DOUBLE) AS temperature_2m,
    CAST(NULL AS DOUBLE) AS precipitation,
    CAST(NULL AS DOUBLE) AS rain,
    CAST(NULL AS DOUBLE) AS snowfall,
    CAST(NULL AS BIGINT) AS weather_code,
    CAST(NULL AS DOUBLE) AS wind_speed_10m,
    'Unknown' AS timezone,
    CAST(NULL AS DOUBLE) AS latitude,
    CAST(NULL AS DOUBLE) AS longitude,
    CAST(NULL AS BIGINT) AS utc_offset_seconds,
    'Unknown' AS source_system,
    CAST(NULL AS STRING) AS source_file,
    CAST(NULL AS STRING) AS batch_id

UNION ALL

SELECT
    CAST(DATE_FORMAT(weather_datetime, 'yyyyMMddHH') AS BIGINT),
    weather_datetime,
    temperature_2m,
    precipitation,
    rain,
    snowfall,
    weather_code,
    wind_speed_10m,
    timezone,
    latitude,
    longitude,
    utc_offset_seconds,
    source_system,
    source_file,
    batch_id
FROM `ftw-week-08`.`02_silver`.weather;

## Create FACT_GREEN_TAXI_TRIP

The fact grain is one accepted Silver Green Taxi source record.

`trip_key` is a deterministic fact-row key generated from stable source business attributes plus `source_file`. It supports repeatable joins and row-level validation, but it is not presented as a proven real-world trip business key.

All accepted Silver rows are retained. The previously tested five-column candidate key is used only for profiling and never for Gold-side deduplication.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.fact_green_taxi_trip
USING DELTA
AS
SELECT
    xxhash64(
        g.VendorID,
        g.lpep_pickup_datetime,
        g.lpep_dropoff_datetime,
        g.store_and_fwd_flag,
        g.RatecodeID,
        g.PULocationID,
        g.DOLocationID,
        g.passenger_count,
        g.trip_distance,
        g.trip_duration_minutes,
        g.fare_amount,
        g.extra,
        g.mta_tax,
        g.tip_amount,
        g.tolls_amount,
        g.improvement_surcharge,
        g.total_amount,
        g.payment_type,
        g.trip_type,
        g.congestion_surcharge,
        g.cbd_congestion_fee,
        g.source_file
    ) AS trip_key,

    g.VendorID AS vendor_id,
    g.lpep_pickup_datetime AS pickup_datetime,
    g.lpep_dropoff_datetime AS dropoff_datetime,

    CAST(DATE_FORMAT(TO_DATE(g.lpep_pickup_datetime), 'yyyyMMdd') AS INT)
        AS pickup_date_key,
    CAST(DATE_FORMAT(TO_DATE(g.lpep_dropoff_datetime), 'yyyyMMdd') AS INT)
        AS dropoff_date_key,

    COALESCE(HOUR(g.lpep_pickup_datetime) + 1, 0) AS pickup_time_key,
    COALESCE(HOUR(g.lpep_dropoff_datetime) + 1, 0) AS dropoff_time_key,

    CAST(g.PULocationID AS INT) AS pickup_taxi_zone_key,
    CAST(g.DOLocationID AS INT) AS dropoff_taxi_zone_key,
    COALESCE(w.weather_hour_key, CAST(0 AS BIGINT)) AS pickup_weather_hour_key,

    CAST(1 AS BIGINT) AS trip_count,
    g.passenger_count,
    g.trip_distance,
    g.trip_duration_minutes,
    CASE
        WHEN g.trip_duration_minutes > 0
        THEN g.trip_distance / (g.trip_duration_minutes / 60.0)
    END AS trip_average_speed_mph,

    g.fare_amount,
    g.extra,
    g.mta_tax,
    g.tip_amount,
    g.tolls_amount,
    g.improvement_surcharge,
    g.congestion_surcharge,
    g.cbd_congestion_fee,
    g.total_amount,

    g.payment_type,
    g.payment_type_description,
    g.trip_type,
    g.RatecodeID AS rate_code_id,
    g.store_and_fwd_flag,

    g.dq_zero_trip_distance,
    g.dq_extreme_trip_distance,
    g.dq_negative_trip_distance,
    g.dq_out_of_range_datetime,
    g.dq_invalid_trip_duration,
    w.weather_hour_key IS NULL AS dq_missing_weather_coverage,

    g.source_system,
    g.source_file,
    g.batch_id

FROM `ftw-week-08`.`02_silver`.green_taxi AS g
LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS w
    ON w.weather_hour_key = CAST(
        DATE_FORMAT(g.pickup_hour, 'yyyyMMddHH') AS BIGINT
    )
   AND w.weather_hour_key != 0;

## Analytics Separation

Business-question analysis is intentionally maintained outside this Gold build notebook in three standalone analytics SQL assets:

- `taxi_demand.sql` — BQ1: demand by day, hour, and zone
- `weather_behavior.sql` — BQ2: Weather and trip behavior
- `area_mobility_patterns.sql` — BQ3: pickup/drop-off area patterns and opportunities

This notebook is limited to Gold dimension/fact creation and technical validation. Keeping analytics separate avoids duplicating business logic inside the mart build.

## Gold Creation Status

The approved Gold dimensions and fact-table build are complete in this notebook. Technical reconciliation, key checks, relationship checks, join-cardinality checks, and rerun comparison are documented in `04_gold_mart_validation.ipynb`.